# Donor Behavioral Clustering — EFA-Informed v2

A deliberately simple clustering notebook built from the factor-analysis findings.

### Default geometry
K-means sees **one standardized score per behavioral unit**, not the raw component features.

1. Transform / coverage-rule / median-impute raw components.
2. Standardize each raw component.
3. Flip signs where needed so every component points in the unit's named direction.
4. Average components **equally within each unit**.
5. Standardize each unit score.
6. Run plain K-means on the unit-score matrix.

**No factor-loading weights, stakeholder weights, PCA, or feature-count weighting.**

The default core uses four units:
- relationship loyalty
- giving rhythm
- amount consistency
- funding posture

The earlier feature groups are retained in an optional-unit library for quick experiments. `campaign_responsiveness` is the first add-on I would test.

**Window note:** the current source build uses explicit `_24m` field names.

In [1]:
from __future__ import annotations

from pathlib import Path
from html import escape
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print(f"pandas={pd.__version__} | numpy={np.__version__}")

pandas=2.3.2 | numpy=1.26.4


## 1 — Configuration

For a baseline run, edit only:
- `MIN_GIFTS_FIELD` / `MIN_GIFTS`
- `EXCLUDE_TEACHERS`
- `ACTIVE_OPTIONAL_UNITS`
- `SELECTED_K`

Examples:

```python
ACTIVE_OPTIONAL_UNITS = []                            # four-unit core
ACTIVE_OPTIONAL_UNITS = ["campaign_responsiveness"]  # preferred first add-on
ACTIVE_OPTIONAL_UNITS = ["jtbd_values_equity"]       # older JTBD sensitivity
```

The notebook blocks any active configuration that reuses the same raw feature in two units, preventing accidental double-counting.

In [2]:
DATA_DIR = Path("/Users/matt.fritz/Desktop/Behavioral Personas/Constructed Data")
FEATURES_PATH = DATA_DIR / "clustering_features_20260801_w24m.csv"

# Population controls.
MIN_GIFTS = 4
MIN_GIFTS_FIELD = "n_gifts_green_24m"   # use "n_gifts_24m" for all project gifts
TOTAL_GIFTS_FIELD = "n_gifts_24m"
EXCLUDE_TEACHERS = False

# K-means controls.
K_VALUES = [3, 4, 5, 6, 7, 8]
SELECTED_K = 7
RANDOM_STATE = 20260825
N_INIT = 20
SILHOUETTE_SAMPLE_N = 5000
STABILITY_RUNS = 10
STABILITY_SAMPLE_FRAC = 0.80

PROJECT_COST_COVERAGE_FIELD = "coverage_project_cost_24m"
PROJECT_COST_COVERAGE_FLOOR = 0.50

# -------------------------------------------------------------------
# CORE: EFA-informed units.
# {feature: direction}; direction is only a sign flip.
# Every component receives equal weight within its unit.
# -------------------------------------------------------------------
ACTIVE_OPTIONAL_UNITS = ["jtbd_local_stewardship","trigger",]

CORE_UNITS = {
    "relationship_loyalty": {
        "share_gifts_repeat_teacher_24m": +1,
        "entropy_school_norm_24m": -1,
    },
    "giving_rhythm": {
        "entropy_gift_month_norm_24m": +1,
        "gifts_per_active_month_24m": -1,
    },
    "giving_approach": {
        "modal_amount_share_24m": +1,
        "share_gifts_round_amount_24m": +1,
        "median_gift_to_project_cost_ratio_24m": -1,
        "share_gifts_closed_project_24m": -1,
        "share_gifts_first_money_in_24m": +1,
    },
}

# -------------------------------------------------------------------
# OPTIONAL LIBRARY.
# Keeps the prior clustering/JTBD groups available for testing.
# Add one idea at a time when possible.
# -------------------------------------------------------------------
OPTIONAL_UNIT_LIBRARY = {
    "campaign_responsiveness": {
        "share_gifts_with_match_24m": +1,
        "share_gifts_big_event_24m": +1,
    },
    "choice_breadth": {
        "top_category_share_count_24m": -1,
    },
    "seasonality": {
        "top_month_share_24m": +1,
        "share_gifts_q4_24m": +1,
    },
    "trigger": {
        "share_gifts_with_match_24m": +1,
        "share_gifts_big_event_24m": +1,
    },
    "funding_depth": {
        "share_gifts_over_half_project_cost_24m": +1,
        "share_gifts_full_project_cost_24m": +1,
    },
    "jtbd_decision_deliberation": {
        "share_gifts_with_prior_search_24m": +1,
        "project_page_pre_gift_mean_24m": +1,
    },
    "jtbd_giving_leverage": {
        "share_gifts_with_match_24m": +1,
        "mean_match_excess_24m": +1,
    },
    "jtbd_sustained_monthly": {
        "is_monthly_donor_current": +1,
        "monthly_longest_streak_months": +1,
    },
    "jtbd_tax_efficiency": {
        "share_gifts_daf_24m": +1,
    },
    "jtbd_active_participation": {
        "sharing_events_24m": +1,
        "sharing_active_months_24m": +1,
    },
    "jtbd_local_stewardship": {
        "share_gifts_within_15mi_24m": +1,
    },
    "jtbd_recognition_avoidance": {
        "share_gifts_anonymous_24m": +1,
    },
    "jtbd_values_equity": {
        "share_gifts_to_low_income_schools_24m": +1,
        "share_gifts_to_historically_underrepresented_race_schools_24m": +1,
    },
    # Extra EFA-informed sensitivities worth keeping nearby.
    "site_engagement": {
        "days_with_site_activity_24m": +1,
        "search_visits_day_total_24m": +1,
    },
    "platform_support": {
        "avg_optional_donation_rate_24m": +1,
        #"share_gifts_with_optional_donation_24m": +1,
    },
}

LOG1P_FIELDS = {
    "gifts_per_active_month_24m",
    "median_gift_to_project_cost_ratio_24m",
    "project_page_pre_gift_mean_24m",
    "mean_match_excess_24m",
    "sharing_events_24m",
    "monthly_longest_streak_months",
    "days_with_site_activity_24m",
    "search_visits_day_total_24m",
}

PROJECT_COST_FIELDS = {
    "median_gift_to_project_cost_ratio_24m",
    "share_gifts_over_half_project_cost_24m",
    "share_gifts_full_project_cost_24m",
}

PROFILE_FIELDS = [
    # scale / value
    "n_gifts_24m", "n_gifts_green_24m", "gift_amount_24m", "median_gift_amount_24m",
    "lifetime_amount", "lifetime_gift_count", "is_major_gift_donor",
    # lifecycle / cadence
    "n_active_months_24m", "days_since_last_gift", "tenure_days",
    "is_new_donor_24m", "is_reactivated_24m", "is_continuing_donor_24m",
    # monthly
    "is_monthly_donor_current", "monthly_active_months_24m",
    "monthly_longest_streak_months", "share_amount_monthly_24m",
    # timing / campaign
    "share_gifts_year_end_final_week_24m", "share_gifts_giving_tuesday_window_24m",
    "share_gifts_back_to_school_24m", "share_gifts_summer_24m", "share_gifts_q4_24m",
    "top_month_share_24m", "share_gifts_with_match_24m", "mean_match_excess_24m",
    "share_gifts_big_event_24m",
    # product / choice / relationship
    "share_gifts_classroom_essentials_24m", "entropy_category_norm_24m",
    "n_unique_categories_24m", "top_category_share_count_24m",
    "n_unique_schools_24m", "share_gifts_repeat_school_24m",
    # funding context
    "median_project_total_cost_24m", "share_gifts_over_half_project_cost_24m",
    "share_gifts_full_project_cost_24m",
    # geography / equity
    "median_distance_mi_24m", "share_gifts_within_15mi_24m",
    "share_gifts_to_low_income_schools_24m",
    "share_gifts_to_historically_underrepresented_race_schools_24m",
    "share_gifts_to_underserved_rural_schools_24m",
    # donor / engagement context
    "is_teacher", "is_teacher_referred", "is_marketing_subscribed",
    "email_open_rate_24m", "email_click_rate_24m",
    "days_with_site_activity_24m", "n_site_visits_24m", "search_visits_day_total_24m",
    "sharing_events_24m", "sharing_active_months_24m",
    "share_gifts_daf_24m", "share_gifts_anonymous_24m", "share_gifts_green_24m",
    # platform support
    "avg_optional_donation_rate_24m", "share_gifts_with_optional_donation_24m",
]

PROFILE_TOP_N = 8
PROFILE_MIN_COVERAGE = 0.50
DEMO_IF_FEATURE_FILE_MISSING = True

assert SELECTED_K in K_VALUES
for name in ACTIVE_OPTIONAL_UNITS:
    if name not in OPTIONAL_UNIT_LIBRARY:
        raise ValueError(f"Unknown optional unit: {name}")

print("Active optional units:", ACTIVE_OPTIONAL_UNITS or "none - four-unit core")

Active optional units: ['jtbd_local_stewardship', 'trigger']


## 2 — Load the donor matrix

Active clustering fields are strict. Inactive optional-library fields and profile fields are loaded when available but do not make the baseline notebook fail.

In [3]:
def active_units(optional_units=None):
    optional_units = ACTIVE_OPTIONAL_UNITS if optional_units is None else list(optional_units)
    units = {name: dict(spec) for name, spec in CORE_UNITS.items()}

    for name in optional_units:
        if name not in OPTIONAL_UNIT_LIBRARY:
            raise ValueError(f"Unknown optional unit: {name}")
        units[name] = dict(OPTIONAL_UNIT_LIBRARY[name])

    seen = {}
    duplicates = {}
    for unit, spec in units.items():
        for feature in spec:
            if feature in seen:
                duplicates.setdefault(feature, [seen[feature]]).append(unit)
            else:
                seen[feature] = unit

    if duplicates:
        msg = "; ".join(f"{f}: {u}" for f, u in duplicates.items())
        raise ValueError(f"Active units reuse the same raw feature. Choose one version of the construct: {msg}")

    return units


def flatten_unit_features(units):
    return [feature for spec in units.values() for feature in spec]


def all_library_features():
    fields = set(flatten_unit_features(CORE_UNITS))
    for spec in OPTIONAL_UNIT_LIBRARY.values():
        fields.update(spec)
    return sorted(fields)


def make_demo_data(n=3500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    latent = rng.choice(5, size=n, p=[0.24, 0.22, 0.20, 0.19, 0.15])

    d = pd.DataFrame({
        "donor_id": [f"demo_{i:05d}" for i in range(n)],
        "cluster_eligible": 1,
        "is_teacher": (rng.random(n) < 0.12).astype(float),
        PROJECT_COST_COVERAGE_FIELD: np.clip(rng.normal(0.90, 0.10, n), 0, 1),
    })

    total_gifts = np.maximum(4, np.rint(np.exp(1.6 + 0.22 * latent + rng.normal(0, 0.35, n))).astype(int))
    green_share = np.clip(0.55 + 0.06 * latent + rng.normal(0, 0.12, n), 0.10, 1.0)
    d["n_gifts_24m"] = total_gifts
    d["n_gifts_green_24m"] = np.rint(total_gifts * green_share).astype(int)

    loyalty = (latent - latent.mean()) / max(latent.std(), 1e-9)
    d["share_gifts_repeat_teacher_24m"] = np.clip(0.55 + 0.14 * loyalty + rng.normal(0, 0.12, n), 0, 1)
    d["entropy_school_norm_24m"] = np.clip(0.55 - 0.15 * loyalty + rng.normal(0, 0.12, n), 0, 1)

    rhythm = rng.normal(size=n)
    d["entropy_gift_month_norm_24m"] = np.clip(0.55 + 0.18 * rhythm + rng.normal(0, 0.10, n), 0, 1)
    d["gifts_per_active_month_24m"] = np.clip(2.7 - 0.70 * rhythm + rng.normal(0, 0.35, n), 1, 7)

    consistency = rng.normal(size=n)
    d["modal_amount_share_24m"] = np.clip(0.55 + 0.18 * consistency + rng.normal(0, 0.10, n), 0, 1)
    d["share_gifts_round_amount_24m"] = np.clip(0.55 + 0.17 * consistency + rng.normal(0, 0.10, n), 0, 1)

    posture = rng.normal(size=n)
    d["median_gift_to_project_cost_ratio_24m"] = np.exp(-2.0 + 0.55 * posture + rng.normal(0, 0.35, n))
    d["share_gifts_closed_project_24m"] = np.clip(0.30 + 0.16 * posture + rng.normal(0, 0.12, n), 0, 1)
    d["share_gifts_first_money_in_24m"] = np.clip(0.45 - 0.14 * posture + rng.normal(0, 0.12, n), 0, 1)

    requested = set(all_library_features()) | set(PROFILE_FIELDS)
    requested |= {MIN_GIFTS_FIELD, TOTAL_GIFTS_FIELD, PROJECT_COST_COVERAGE_FIELD}

    for f in requested:
        if f in d.columns:
            continue
        name = f.lower()
        if name.startswith(("share_", "is_", "has_")) or "_rate" in name or "entropy_" in name:
            d[f] = rng.beta(2, 4, n)
        elif "amount" in name and "share_" not in name and "_rate" not in name:
            d[f] = np.exp(rng.normal(4.0, 0.7, n))
        elif "ratio" in name or "match_excess" in name:
            d[f] = np.exp(rng.normal(-1.5, 0.6, n))
        elif any(token in name for token in ["days", "months", "count", "n_unique", "visits", "events", "payments"]):
            d[f] = np.maximum(0, np.rint(np.exp(rng.normal(1.5, 0.6, n))))
        else:
            d[f] = rng.normal(0, 1, n)

    for f, rate in {
        "median_gift_to_project_cost_ratio_24m": 0.06,
        "median_distance_mi_24m": 0.30,
        "share_gifts_within_15mi_24m": 0.30,
    }.items():
        if f in d.columns:
            d.loc[rng.random(n) < rate, f] = np.nan

    return d


UNITS_NOW = active_units()
FIT_FIELDS_NOW = flatten_unit_features(UNITS_NOW)

control_fields = [
    "donor_id", "cluster_eligible", MIN_GIFTS_FIELD, TOTAL_GIFTS_FIELD,
    "is_teacher", PROJECT_COST_COVERAGE_FIELD,
]
candidate_fields = list(dict.fromkeys(
    control_fields + FIT_FIELDS_NOW + all_library_features() + PROFILE_FIELDS
))

if FEATURES_PATH.exists():
    header = pd.read_csv(FEATURES_PATH, nrows=0).columns.tolist()
    header_set = set(header)

    required = ["donor_id", "cluster_eligible", MIN_GIFTS_FIELD] + FIT_FIELDS_NOW
    if EXCLUDE_TEACHERS:
        required.append("is_teacher")

    missing_required = [f for f in required if f not in header_set]
    if missing_required:
        raise ValueError(
            "Required active fields missing from feature CSV:\n"
            + "\n".join(f"  - {f}" for f in missing_required)
        )

    usecols = [f for f in candidate_fields if f in header_set]
    FEATURES = pd.read_csv(FEATURES_PATH, usecols=usecols)
    DATA_SOURCE = str(FEATURES_PATH)
else:
    if not DEMO_IF_FEATURE_FILE_MISSING:
        raise FileNotFoundError(FEATURES_PATH)
    FEATURES = make_demo_data()
    DATA_SOURCE = "SYNTHETIC SMOKE TEST - real donor-level feature CSV is not attached here"

if FEATURES["donor_id"].duplicated().any():
    raise ValueError("donor_id must be unique")

eligibility = pd.to_numeric(FEATURES[MIN_GIFTS_FIELD], errors="coerce")
mask = pd.to_numeric(FEATURES["cluster_eligible"], errors="coerce").eq(1) & eligibility.ge(MIN_GIFTS)

if EXCLUDE_TEACHERS:
    mask &= ~pd.to_numeric(FEATURES["is_teacher"], errors="coerce").eq(1)

DONORS = FEATURES.loc[mask].copy().set_index("donor_id")

if len(DONORS) <= SELECTED_K:
    raise ValueError(f"Only {len(DONORS)} eligible donors; not enough for K={SELECTED_K}")

print(f"Data source: {DATA_SOURCE}")
print(f"Eligible donors: {len(DONORS):,} | {MIN_GIFTS_FIELD} >= {MIN_GIFTS} | EXCLUDE_TEACHERS={EXCLUDE_TEACHERS}")
print(f"Active units ({len(UNITS_NOW)}): {list(UNITS_NOW)}")
print(f"Raw component fields ({len(FIT_FIELDS_NOW)}): {FIT_FIELDS_NOW}")

Data source: /Users/matt.fritz/Desktop/Behavioral Personas/Constructed Data/clustering_features_20260801_w24m.csv
Eligible donors: 153,366 | n_gifts_green_24m >= 2 | EXCLUDE_TEACHERS=True
Active units (5): ['relationship_loyalty', 'giving_rhythm', 'giving_approach', 'jtbd_local_stewardship', 'trigger']
Raw component fields (12): ['share_gifts_repeat_teacher_24m', 'entropy_school_norm_24m', 'entropy_gift_month_norm_24m', 'gifts_per_active_month_24m', 'modal_amount_share_24m', 'share_gifts_round_amount_24m', 'median_gift_to_project_cost_ratio_24m', 'share_gifts_closed_project_24m', 'share_gifts_first_money_in_24m', 'share_gifts_within_15mi_24m', 'share_gifts_with_match_24m', 'share_gifts_big_event_24m']


## 3 — Sanity-check the active raw components

This is only a guardrail for missing, degenerate, or extremely discrete component features before they are collapsed into unit scores.

In [4]:
def feature_sanity_table(df, units):
    feature_to_unit = {f: unit for unit, spec in units.items() for f in spec}
    direction_lookup = {f: direction for spec in units.values() for f, direction in spec.items()}
    rows = []

    for f in flatten_unit_features(units):
        x = pd.to_numeric(df[f], errors="coerce")
        observed = x.dropna()
        rows.append({
            "unit": feature_to_unit[f],
            "feature": f,
            "direction": "+" if direction_lookup[f] > 0 else "-",
            "missing": x.isna().mean(),
            "n_unique": observed.nunique(),
            "std": observed.std(ddof=0),
            "p10": observed.quantile(0.10) if len(observed) else np.nan,
            "p50": observed.quantile(0.50) if len(observed) else np.nan,
            "p90": observed.quantile(0.90) if len(observed) else np.nan,
            "% zero": (observed == 0).mean() if len(observed) else np.nan,
            "% one": (observed == 1).mean() if len(observed) else np.nan,
            "transform": "log1p" if f in LOG1P_FIELDS else "none",
        })

    return pd.DataFrame(rows)


SANITY = feature_sanity_table(DONORS, UNITS_NOW)
display(
    SANITY.style
    .format({
        "missing": "{:.1%}", "std": "{:.3f}", "p10": "{:.3f}",
        "p50": "{:.3f}", "p90": "{:.3f}", "% zero": "{:.1%}", "% one": "{:.1%}"
    })
    .hide(axis="index")
    .set_caption("Active raw-component sanity check")
)

bad = SANITY.loc[(SANITY["n_unique"] <= 1) | (SANITY["std"].fillna(0) <= 1e-12), "feature"].tolist()
if bad:
    raise ValueError(f"Degenerate active fit features: {bad}")

unit,feature,direction,missing,n_unique,std,p10,p50,p90,% zero,% one,transform
relationship_loyalty,share_gifts_repeat_teacher_24m,+,0.2%,2175,0.334,0.000,0.333,1.000,34.4%,11.0%,none
relationship_loyalty,entropy_school_norm_24m,-,9.5%,6285,0.437,0.000,0.000,1.000,50.3%,19.8%,none
giving_rhythm,entropy_gift_month_norm_24m,+,0.0%,9956,0.388,0.000,0.693,1.000,22.4%,26.6%,none
giving_rhythm,gifts_per_active_month_24m,-,0.0%,1662,2.788,1.000,1.500,3.000,0.0%,40.7%,log1p
giving_approach,modal_amount_share_24m,+,0.0%,2003,0.270,0.222,0.500,1.000,0.0%,20.4%,none
giving_approach,share_gifts_round_amount_24m,+,0.0%,2271,0.359,0.000,0.750,1.000,10.2%,44.5%,none
giving_approach,median_gift_to_project_cost_ratio_24m,-,0.8%,142643,0.185,0.029,0.097,0.428,0.0%,0.6%,log1p
giving_approach,share_gifts_closed_project_24m,-,0.0%,2047,0.218,0.000,0.000,0.500,64.6%,1.4%,none
giving_approach,share_gifts_first_money_in_24m,+,0.0%,1414,0.206,0.000,0.000,0.500,64.3%,1.1%,none
jtbd_local_stewardship,share_gifts_within_15mi_24m,+,33.1%,1543,0.444,0.000,0.900,1.000,29.2%,48.7%,none


## 4 — Build one equally weighted score per behavioral unit

**raw feature → transform / coverage rule → median impute → feature z-score → sign flip → equal mean within unit → unit z-score**

K-means only sees the final unit-score matrix.

In [5]:
def prepare_unit_matrix(df, units):
    fields = flatten_unit_features(units)
    X = df[fields].apply(pd.to_numeric, errors="coerce").copy()

    if PROJECT_COST_COVERAGE_FIELD in df.columns:
        cov = pd.to_numeric(df[PROJECT_COST_COVERAGE_FIELD], errors="coerce")
        low_cov = cov < PROJECT_COST_COVERAGE_FLOOR
        for f in PROJECT_COST_FIELDS.intersection(fields):
            X.loc[low_cov, f] = np.nan

    for f in fields:
        if f in LOG1P_FIELDS:
            if (X[f].dropna() < 0).any():
                raise ValueError(f"{f} has negative values but is declared log1p")
            X[f] = np.log1p(X[f])

    missing_before = X.isna().mean()
    medians = X.median(numeric_only=True)
    all_missing = medians[medians.isna()].index.tolist()
    if all_missing:
        raise ValueError(f"Active fit fields are entirely missing after coverage rules: {all_missing}")

    X_imp = X.fillna(medians)

    feature_scaler = StandardScaler()
    feature_z = pd.DataFrame(
        feature_scaler.fit_transform(X_imp),
        index=X_imp.index,
        columns=X_imp.columns,
    )

    unit_raw = pd.DataFrame(index=X_imp.index)
    for unit, spec in units.items():
        signed = pd.DataFrame(
            {f: feature_z[f] * direction for f, direction in spec.items()},
            index=X_imp.index,
        )
        unit_raw[unit] = signed.mean(axis=1)

    unit_scaler = StandardScaler()
    unit_z = pd.DataFrame(
        unit_scaler.fit_transform(unit_raw),
        index=unit_raw.index,
        columns=unit_raw.columns,
    )

    prep = {
        "fields": fields,
        "units": units,
        "medians": medians,
        "missing_before": missing_before,
        "feature_scaler": feature_scaler,
        "unit_scaler": unit_scaler,
        "feature_z": feature_z,
        "unit_raw": unit_raw,
    }
    return unit_z, prep


Z, PREP = prepare_unit_matrix(DONORS, UNITS_NOW)

quality = pd.DataFrame({
    "feature": PREP["fields"],
    "missing_before_impute": [PREP["missing_before"][f] for f in PREP["fields"]],
})
display(
    quality.style
    .format({"missing_before_impute": "{:.1%}"})
    .hide(axis="index")
    .set_caption("Fit matrix ready")
)

display(
    Z.corr().style
    .format("{:+.2f}")
    .set_caption("Correlation among final unit scores")
)

print(f"K-means matrix: {Z.shape[0]:,} donors x {Z.shape[1]} equally weighted units")

feature,missing_before_impute
share_gifts_repeat_teacher_24m,0.2%
entropy_school_norm_24m,9.5%
entropy_gift_month_norm_24m,0.0%
gifts_per_active_month_24m,0.0%
modal_amount_share_24m,0.0%
share_gifts_round_amount_24m,0.0%
median_gift_to_project_cost_ratio_24m,6.7%
share_gifts_closed_project_24m,0.0%
share_gifts_first_money_in_24m,0.0%
share_gifts_within_15mi_24m,33.1%


,relationship_loyalty,giving_rhythm,giving_approach,jtbd_local_stewardship,trigger
relationship_loyalty,+1.00,+0.22,+0.18,+0.02,+0.13
giving_rhythm,+0.22,+1.00,+0.34,+0.07,+0.09
giving_approach,+0.18,+0.34,+1.00,+0.12,+0.05
jtbd_local_stewardship,+0.02,+0.07,+0.12,+1.00,+0.01
trigger,+0.13,+0.09,+0.05,+0.01,+1.00


K-means matrix: 153,366 donors x 5 equally weighted units


## 5 — K diagnostics

Use silhouette and cluster balance to compare **K within this exact unit space**. They do not choose the strategic winner automatically.

In [6]:
def canonicalize_labels(labels):
    counts = pd.Series(labels).value_counts().sort_values(ascending=False)
    mapping = {old: new for new, old in enumerate(counts.index, start=1)}
    return np.array([mapping[x] for x in labels], dtype=int)


def subsample_stability_ari(
    Z,
    k,
    runs=STABILITY_RUNS,
    sample_frac=STABILITY_SAMPLE_FRAC,
):
    """
    Fit K-means repeatedly on random donor subsamples.
    Each fitted model then assigns every donor.
    Pairwise ARI measures how consistently the donor partition reappears.
    """
    X = Z.to_numpy()
    n = len(X)
    sample_n = max(k + 1, int(n * sample_frac))

    rng = np.random.default_rng(RANDOM_STATE + 1000 * k)
    predictions = []

    for run in range(runs):
        idx = rng.choice(n, size=sample_n, replace=False)

        km = KMeans(
            n_clusters=k,
            n_init=N_INIT,
            random_state=RANDOM_STATE + 1000 * k + run,
        )
        km.fit(X[idx])

        # Predict all donors so every run is compared on the same population.
        predictions.append(km.predict(X))

    aris = []
    for i in range(len(predictions)):
        for j in range(i + 1, len(predictions)):
            aris.append(
                adjusted_rand_score(predictions[i], predictions[j])
            )

    aris = np.asarray(aris)

    return {
        "stability_ARI_mean": aris.mean(),
        "stability_ARI_p10": np.quantile(aris, 0.10),
    }


def k_diagnostics(Z, k_values=K_VALUES):
    rows = []
    X = Z.to_numpy()

    for k in k_values:
        if k >= len(X):
            continue

        km = KMeans(
            n_clusters=k,
            n_init=N_INIT,
            random_state=RANDOM_STATE,
        )
        labels = canonicalize_labels(km.fit_predict(X))

        counts = pd.Series(labels).value_counts(normalize=True)

        if len(X) > SILHOUETTE_SAMPLE_N:
            sil = silhouette_score(
                X,
                labels,
                sample_size=SILHOUETTE_SAMPLE_N,
                random_state=RANDOM_STATE,
            )
            sil_n = SILHOUETTE_SAMPLE_N
        else:
            sil = silhouette_score(X, labels)
            sil_n = len(X)

        stability = subsample_stability_ari(Z, k)

        rows.append({
            "K": k,
            "silhouette": sil,
            "stability_ARI_mean": stability["stability_ARI_mean"],
            "stability_ARI_p10": stability["stability_ARI_p10"],
            "smallest_cluster": counts.min(),
            "largest_cluster": counts.max(),
            "inertia_per_donor": km.inertia_ / len(X),
            "silhouette_n": sil_n,
        })

    return pd.DataFrame(rows)


K_DIAGNOSTICS = k_diagnostics(Z)

display(
    K_DIAGNOSTICS.style
    .format({
        "silhouette": "{:.3f}",
        "stability_ARI_mean": "{:.3f}",
        "stability_ARI_p10": "{:.3f}",
        "smallest_cluster": "{:.1%}",
        "largest_cluster": "{:.1%}",
        "inertia_per_donor": "{:.3f}",
        "silhouette_n": "{:,.0f}",
    })
    .hide(axis="index")
    .set_caption(
        "K diagnostics — silhouette = separation; "
        "stability ARI = reproducibility across 80% donor subsamples"
    )
)

K,silhouette,stability_ARI_mean,stability_ARI_p10,smallest_cluster,largest_cluster,inertia_per_donor,silhouette_n
3,0.229,0.991,0.987,18.8%,47.4%,3.323,"5,000"
4,0.213,0.986,0.978,17.1%,37.3%,2.871,"5,000"
5,0.213,0.944,0.742,14.8%,27.8%,2.560,"5,000"
6,0.229,0.995,0.993,11.8%,23.0%,2.275,"5,000"
7,0.228,0.987,0.977,10.2%,21.1%,2.098,"5,000"
8,0.227,0.992,0.988,6.4%,19.6%,1.962,"5,000"


## 6 — Fit the selected K

Cluster numbers are ordered largest-to-smallest for readability only.

In [7]:
def fit_selected(Z, k=SELECTED_K):
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE)
    raw_labels = km.fit_predict(Z.to_numpy())
    labels = canonicalize_labels(raw_labels)
    assignments = pd.DataFrame({"cluster": labels}, index=Z.index)
    return km, assignments


MODEL, ASSIGNMENTS = fit_selected(Z, SELECTED_K)
DONORS_WITH_CLUSTER = DONORS.join(ASSIGNMENTS)
UNIT_SCORES_WITH_CLUSTER = Z.join(ASSIGNMENTS)

cluster_sizes = (
    ASSIGNMENTS["cluster"].value_counts().sort_index().rename("n").to_frame()
    .assign(share=lambda x: x["n"] / x["n"].sum())
)
display(
    cluster_sizes.style
    .format({"n": "{:,.0f}", "share": "{:.1%}"})
    .set_caption(f"Selected solution: K={SELECTED_K}")
)

,n,share
cluster,,
1,"32,420",21.1%
2,"23,894",15.6%
3,"23,648",15.4%
4,"22,242",14.5%
5,"17,790",11.6%
6,"17,781",11.6%
7,"15,591",10.2%


## 7 — Clean cluster profiles

Each cluster card has three levels:

1. **Behavioral dimensions** — the unit scores K-means actually saw.
2. **Raw components** — the observable behaviors behind those scores.
3. **Strongest profile-only differences** — context that did not influence the fit.

Distinctiveness is descriptive, not causal importance.

In [8]:
def is_rate_like(field, series):
    name = field.lower()
    observed = pd.to_numeric(series, errors="coerce").dropna()
    bounded = len(observed) and observed.min() >= -1e-9 and observed.max() <= 1.000001
    return name.startswith(("share_", "is_", "has_")) or "_rate" in name or bounded


def raw_summary(field, series):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if not len(x):
        return np.nan, "mean"
    if is_rate_like(field, x):
        return float(x.mean()), "mean"
    return float(x.median()), "median"


def format_raw(field, value, method, reference_series):
    if pd.isna(value):
        return "-"
    name = field.lower()
    if is_rate_like(field, reference_series):
        return f"{value:.0%}"
    if "amount" in name and "share_" not in name and "rate" not in name and "ratio" not in name:
        return f"${value:,.0f}"
    if "ratio" in name:
        return f"{value:.2f}x"
    if any(token in name for token in ["days", "months", "count", "n_unique", "n_gifts", "visits", "events"]):
        return f"{value:,.1f}"
    return f"{value:,.2f}"


def build_feature_profile_long(df_with_cluster, units, profile_fields=PROFILE_FIELDS):
    fit_fields = flatten_unit_features(units)
    field_to_unit = {f: unit for unit, spec in units.items() for f in spec}
    available_profile = [f for f in profile_fields if f in df_with_cluster.columns and f not in fit_fields]
    fields = list(dict.fromkeys(fit_fields + available_profile))

    numeric = df_with_cluster[fields].apply(pd.to_numeric, errors="coerce")
    overall_mean = numeric.mean()
    overall_std = numeric.std(ddof=0).replace(0, np.nan)
    coverage = numeric.notna().mean()

    overall_raw = {}
    methods = {}
    for f in fields:
        overall_raw[f], methods[f] = raw_summary(f, numeric[f])

    rows = []
    for cluster, idx in df_with_cluster.groupby("cluster").groups.items():
        sub = numeric.loc[idx]
        effect = (sub.mean() - overall_mean) / overall_std

        for f in fields:
            cv, _ = raw_summary(f, sub[f])
            rows.append({
                "cluster": int(cluster),
                "feature": f,
                "source": "COMPONENT" if f in fit_fields else "PROFILE",
                "unit": field_to_unit.get(f, "profile-only"),
                "cluster_value": cv,
                "overall_value": overall_raw[f],
                "raw_method": methods[f],
                "effect_z": float(effect[f]) if pd.notna(effect[f]) else np.nan,
                "abs_effect_z": float(abs(effect[f])) if pd.notna(effect[f]) else np.nan,
                "coverage": float(coverage[f]),
            })

    return pd.DataFrame(rows)


FEATURE_PROFILE_LONG = build_feature_profile_long(DONORS_WITH_CLUSTER, UNITS_NOW)


def build_unit_profile_long(unit_scores_with_cluster):
    rows = []
    unit_cols = [c for c in unit_scores_with_cluster.columns if c != "cluster"]

    for cluster, sub in unit_scores_with_cluster.groupby("cluster"):
        for unit in unit_cols:
            value = float(sub[unit].mean())
            rows.append({
                "cluster": int(cluster),
                "unit": unit,
                "score": value,
                "abs_score": abs(value),
            })

    return pd.DataFrame(rows)


UNIT_PROFILE_LONG = build_unit_profile_long(UNIT_SCORES_WITH_CLUSTER)


def effect_bar(z):
    if pd.isna(z):
        return '<span style="color:#777">-</span>'

    width = min(abs(float(z)) / 2.0, 1.0) * 100
    fill = "#dbeafe" if z > 0 else "#fee2e2"
    label = f"{z:+.2f} SD"

    return (
        '<div style="min-width:145px">'
        '<div style="height:18px;background:#f3f4f6;border-radius:3px;position:relative;overflow:hidden">'
        f'<div style="height:100%;width:{width:.1f}%;background:{fill}"></div>'
        f'<span style="position:absolute;left:6px;top:1px;font-size:12px;font-weight:600">{label}</span>'
        '</div></div>'
    )


def unit_table_html(rows):
    body = []
    for r in rows.itertuples(index=False):
        body.append(
            "<tr>"
            f"<td style='font-size:13px;font-weight:600'>{escape(str(r.unit).replace('_', ' '))}</td>"
            f"<td style='text-align:right;font-weight:600'>{r.score:+.2f}</td>"
            f"<td>{effect_bar(r.score)}</td>"
            "</tr>"
        )

    return (
        "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
        "<thead><tr style='background:#f8fafc'>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Behavioral dimension</th>"
        "<th style='text-align:right;padding:6px;border-bottom:1px solid #ddd'>Mean score</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Distinctiveness</th>"
        "</tr></thead><tbody>"
        + "".join(body)
        + "</tbody></table>"
    )


def feature_table_html(rows, df_reference):
    body = []

    for r in rows.itertuples(index=False):
        cv = format_raw(r.feature, r.cluster_value, r.raw_method, df_reference[r.feature])
        ov = format_raw(r.feature, r.overall_value, r.raw_method, df_reference[r.feature])

        body.append(
            "<tr>"
            f"<td style='font-family:ui-monospace,monospace;font-size:12px'>{escape(r.feature)}</td>"
            f"<td style='font-size:12px;color:#555'>{escape(str(r.unit))}</td>"
            f"<td style='text-align:right;font-weight:600'>{cv}</td>"
            f"<td style='text-align:right;color:#666'>{ov}</td>"
            f"<td>{effect_bar(r.effect_z)}</td>"
            "</tr>"
        )

    return (
        "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
        "<thead><tr style='background:#f8fafc'>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Feature</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Unit</th>"
        "<th style='text-align:right;padding:6px;border-bottom:1px solid #ddd'>Cluster</th>"
        "<th style='text-align:right;padding:6px;border-bottom:1px solid #ddd'>Overall</th>"
        "<th style='text-align:left;padding:6px;border-bottom:1px solid #ddd'>Distinctiveness</th>"
        "</tr></thead><tbody>"
        + "".join(body)
        + "</tbody></table>"
    )


def show_cluster_profiles(
    unit_profile=UNIT_PROFILE_LONG,
    feature_profile=FEATURE_PROFILE_LONG,
    df_reference=DONORS_WITH_CLUSTER,
    top_profile_n=PROFILE_TOP_N,
):
    total_n = len(df_reference)

    for cluster in sorted(feature_profile["cluster"].unique()):
        n = int((df_reference["cluster"] == cluster).sum())

        unit_rows = unit_profile.loc[
            unit_profile["cluster"] == cluster
        ].sort_values("abs_score", ascending=False)

        component_rows = feature_profile.loc[
            (feature_profile["cluster"] == cluster)
            & (feature_profile["source"] == "COMPONENT")
        ].sort_values(["unit", "abs_effect_z"], ascending=[True, False])

        profile_rows = feature_profile.loc[
            (feature_profile["cluster"] == cluster)
            & (feature_profile["source"] == "PROFILE")
            & (feature_profile["coverage"] >= PROFILE_MIN_COVERAGE)
        ].sort_values("abs_effect_z", ascending=False).head(top_profile_n)

        html = f"""
        <div style='border:1px solid #d1d5db;border-radius:8px;padding:14px 16px;margin:14px 0 22px 0;background:white'>
          <div style='font-size:21px;font-weight:700;margin-bottom:2px'>Cluster {cluster}</div>
          <div style='color:#555;margin-bottom:14px'>{n:,} donors - {n/total_n:.1%} of clustered population</div>

          <div style='font-size:15px;font-weight:700;margin:8px 0'>Behavioral dimensions used by K-means</div>
          {unit_table_html(unit_rows)}

          <div style='font-size:15px;font-weight:700;margin:16px 0 8px 0'>Raw components behind those dimensions</div>
          {feature_table_html(component_rows, df_reference)}

          <div style='font-size:15px;font-weight:700;margin:16px 0 8px 0'>Strongest profile-only differences</div>
          {feature_table_html(profile_rows, df_reference)}
        </div>
        """

        display(HTML(html))


show_cluster_profiles()

## 8 — Optional experiments

The baseline stays untouched. Add one or more retained optional units and compare assignments with the baseline using ARI.

**ARI measures assignment agreement, not model quality.**

In [9]:
def run_experiment(optional_units, k=SELECTED_K):
    units = active_units(optional_units)
    required = flatten_unit_features(units)
    missing = [f for f in required if f not in DONORS.columns]

    if missing:
        raise ValueError(f"Experiment requires fields that were not available/loaded: {missing}")

    Z_alt, prep_alt = prepare_unit_matrix(DONORS, units)
    km_alt = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE)
    labels_alt = canonicalize_labels(km_alt.fit_predict(Z_alt.to_numpy()))
    assignments_alt = pd.DataFrame({"cluster": labels_alt}, index=Z_alt.index)

    return {
        "optional_units": list(optional_units),
        "units": units,
        "Z": Z_alt,
        "prep": prep_alt,
        "model": km_alt,
        "assignments": assignments_alt,
    }


def compare_to_core(optional_units, k=SELECTED_K):
    alt = run_experiment(optional_units, k=k)
    common = ASSIGNMENTS.index.intersection(alt["assignments"].index)

    ari = adjusted_rand_score(
        ASSIGNMENTS.loc[common, "cluster"],
        alt["assignments"].loc[common, "cluster"],
    )

    return pd.DataFrame([{
        "experiment": "core + " + " + ".join(optional_units),
        "K": k,
        "units": len(alt["units"]),
        "ARI_vs_core": ari,
        "same_donors_n": len(common),
    }])


# Examples:
# display(compare_to_core(["campaign_responsiveness"]))
# display(compare_to_core(["jtbd_values_equity"]))
# display(compare_to_core(["site_engagement"]))

print("Optional experiment helpers are ready.")
print("Available optional units:")
print(sorted(OPTIONAL_UNIT_LIBRARY))

Optional experiment helpers are ready.
Available optional units:
['campaign_responsiveness', 'choice_breadth', 'funding_depth', 'jtbd_active_participation', 'jtbd_decision_deliberation', 'jtbd_giving_leverage', 'jtbd_local_stewardship', 'jtbd_recognition_avoidance', 'jtbd_sustained_monthly', 'jtbd_tax_efficiency', 'jtbd_values_equity', 'platform_support', 'seasonality', 'site_engagement', 'trigger']


## 9 — Optional export

Useful objects:
- `ASSIGNMENTS` — donor → cluster
- `Z` — standardized unit scores K-means actually used
- `UNIT_PROFILE_LONG` — cluster profiles on behavioral dimensions
- `FEATURE_PROFILE_LONG` — raw components + profile-only fields
- `K_DIAGNOSTICS` — K diagnostics

In [10]:
# EXPORT_DIR = DATA_DIR.parent / "Clustering Runs - EFA Informed"
# EXPORT_DIR.mkdir(parents=True, exist_ok=True)
# ASSIGNMENTS.reset_index().to_csv(EXPORT_DIR / "cluster_assignments.csv", index=False)
# Z.reset_index().to_csv(EXPORT_DIR / "unit_scores.csv", index=False)
# UNIT_PROFILE_LONG.to_csv(EXPORT_DIR / "unit_profiles.csv", index=False)
# FEATURE_PROFILE_LONG.to_csv(EXPORT_DIR / "feature_profiles.csv", index=False)
# K_DIAGNOSTICS.to_csv(EXPORT_DIR / "k_diagnostics.csv", index=False)
# print(f"Wrote outputs to {EXPORT_DIR}")

print("Nothing exported by default.")

Nothing exported by default.


In [11]:
# ============================================================================
# Behavioral JTBD profiling — donor-level ranking + cluster summaries
#
# Interpretation:
# These are behavioral signals CONSISTENT WITH a Job, not proof that the donor
# consciously holds that motivation.
#
# IMPORTANT:
# - Top-N comparisons use COMPLETE CASES across all rankable Jobs.
# - Evidence-only Jobs are scored but cannot consume Top-N slots.
# - A Job is called a "standout" only when both relative AND absolute evidence
#   clear the thresholds below.
# ============================================================================

import numpy as np
import pandas as pd

from IPython.display import HTML, Markdown, display


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------

JTBD_TOP_N = 2

# Otherwise-rankable Job must be measurable for at least this share of donors.
JTBD_MIN_JOB_COVERAGE = 0.55

# Business-facing standout rules.
# These are display / interpretation rules, not statistical significance tests.
JTBD_STANDOUT_MIN_INDEX = 1.25
JTBD_STANDOUT_MIN_EVIDENCE_Z = 0.20

# Maximum standout Jobs shown on each cluster card.
JTBD_SHOW_TOP_PER_CLUSTER = 5


# ----------------------------------------------------------------------------
# JOB DEFINITIONS
#
# Feature value = signed within-Job weight.
#
# Positive = more feature -> more evidence for Job
# Negative = less feature -> more evidence for Job
#
# Each completed Job score is standardized across donors before Jobs are
# compared within a donor.
# ----------------------------------------------------------------------------

JTBD_JOB_SPECS = {

    # ------------------------------------------------------------------------
    # FUNCTIONAL
    # ------------------------------------------------------------------------

    "Concrete impact": {
        "definition": "Produce a tangible, understandable result",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Keep only one direct funding-depth measure so this does not
            # simply become "large giver."
            "median_gift_to_project_cost_ratio_24m": +0.75,

            # Different manifestation: finishing a concrete project need.
            "share_gifts_closed_project_24m": +1.00,

            # Tangible item/list giving; useful but product-opportunity dependent.
            "share_gifts_classroom_essentials_24m": +0.50,
        },
    },

    "Directed choice": {
        "definition": "Understand the specific project or destination my gift will support",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_with_prior_search_24m": +1.00,
            "project_page_pre_gift_mean_24m": +1.00,
        },
    },

    "Giving leverage": {
        "definition": "Make my contribution accomplish more by unlocking additional funds",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_with_match_24m": +1.00,
            "mean_match_excess_24m": +1.00,
        },
    },

    "Tax efficiency": {
        "definition": "Leverage charitable tax deductions for my financial benefit",

        # Useful directional evidence, but not strong enough to compete
        # against better-observed Jobs for a donor's Top-2 slots.
        "rankable": False,
        "min_components": 2,
        "features": {
            "share_gifts_daf_24m": +2.00,
            "share_gifts_year_end_final_week_24m": +1.00,
            "share_gifts_q4_24m": +0.50,
        },
    },

    "Sustained giving": {
        "definition": "Make generosity a dependable practice",
        "rankable": True,
        "min_components": 3,
        "features": {
            # Recent consistency.
            "n_active_months_24m": +1.00,
            "entropy_gift_month_norm_24m": +1.00,

            # Persistence across windows.
            "is_continuing_donor_24m": +1.00,

            # Length of overall giving relationship.
            "tenure_days_any_giving": +0.75,

            # Only one explicitly monthly-specific measure.
            "monthly_longest_streak_months": +0.50,
        },
    },

    "Confidence and risk reduction": {
        "definition": "Avoid an unsafe, ineffective, or regrettable giving decision",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Repeated school support as an observed trust/persistence signal.
            "share_gifts_repeat_school_24m": +1.00,

            # Research / deliberation.
            "share_gifts_with_prior_search_24m": +0.75,
            "teacher_page_pre_gift_mean_24m": +0.50,
        },
    },

    # ------------------------------------------------------------------------
    # SOCIAL
    # ------------------------------------------------------------------------

    "Relational support": {
        "definition": "Show up for someone I care about",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_repeat_teacher_24m": +1.00,
            "top_teacher_share_count_24m": +1.00,
            "gifts_per_teacher_24m": +0.75,

            # Relationship-origin context, weaker than observed repeated support.
            "is_teacher_referred": +0.50,
        },
    },

    "Local stewardship": {
        "definition": "Strengthen the community where I live",
        "rankable": True,
        "min_components": 2,
        "features": {
            "share_gifts_within_15mi_24m": +1.00,
            "median_distance_mi_24m": -1.00,
            "share_gifts_same_state_24m": +0.75,
        },
    },

    "Collective participation": {
        "definition": "Join others I trust in supporting something together",

        # We can see some relevant behavior, but sharing/referral evidence is
        # currently too sparse/contextual to compete for Top-2 slots.
        "rankable": False,
        "min_components": 2,
        "features": {
            "sharing_active_months_24m": +1.00,
            "share_gifts_channel_sharetray_24m": +1.00,
            "share_gifts_channel_facebook_24m": +0.50,
            "share_gifts_channel_nextdoor_24m": +0.50,
        },
    },

    # ------------------------------------------------------------------------
    # EMOTIONAL / CROSS-CUTTING
    # ------------------------------------------------------------------------

    "Urgent action": {
        "definition": "Act when a need becomes urgent and emotionally real",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Direct project urgency.
            "days_gift_to_expiration_median_24m": -1.00,
            "share_gifts_expiring_soon_24m": +1.00,

            # Supporting late-cycle evidence.
            "share_gifts_late_cycle_24m": +0.50,

            # Separate manifestation: speed of action after encountering need.
            "share_gifts_same_day_first_session_24m": +0.50,
            "days_first_session_to_gift_median_24m": -0.50,
        },
    },

    "Values obligation": {
        "definition": "Feel true to my personal values by giving",
        "rankable": True,
        "min_components": 2,
        "features": {
            # Specifically equity-oriented behavioral expressions of Values.
            "share_gifts_to_low_income_schools_24m": +1.00,
            "share_gifts_to_historically_underrepresented_race_schools_24m": +1.00,
            "share_gifts_to_underserved_rural_schools_24m": +1.00,
        },
    },
}


# Jobs where current 1P data cannot support a defensible behavioral score.
JTBD_NOT_RANKED = {

    "Impact witness":
        "No clean post-gift sequence showing that the donor actually follows "
        "the work or consumes evidence of progress.",

    "Active participation":
        "Current first-party data do not directly observe taking part in the "
        "work beyond giving money. Sharing is treated as Collective "
        "Participation instead.",

    "Acknowledged contribution":
        "Anonymous giving is a narrow inverse proxy, but is not enough to "
        "support a multi-signal behavioral score for desire for recognition.",

    "Faith obligation":
        "No defensible behavioral signal in the current first-party feature build.",
}


# ----------------------------------------------------------------------------
# 1. BUILD PROFILING FRAME
# ----------------------------------------------------------------------------

if "DONORS_WITH_CLUSTER" not in globals():
    raise NameError(
        "Run the clustering notebook through DONORS_WITH_CLUSTER before this cell."
    )

JTBD_BASE = DONORS_WITH_CLUSTER.copy()

if JTBD_BASE.index.name != "donor_id" and "donor_id" in JTBD_BASE.columns:
    JTBD_BASE = JTBD_BASE.set_index("donor_id")


_all_jtbd_fields = sorted({
    f
    for spec in JTBD_JOB_SPECS.values()
    for f in spec["features"]
})

_control_fields = [
    "coverage_project_cost_24m",
]

_missing = [
    f
    for f in _all_jtbd_fields + _control_fields
    if f not in JTBD_BASE.columns
]


# Pull missing JTBD fields from original feature CSV when possible.
if (
    _missing
    and "FEATURES_PATH" in globals()
    and FEATURES_PATH.exists()
):

    _header = set(
        pd.read_csv(
            FEATURES_PATH,
            nrows=0,
        ).columns
    )

    _loadable = [
        f for f in _missing
        if f in _header
    ]

    if _loadable:
        _extra = (
            pd.read_csv(
                FEATURES_PATH,
                usecols=["donor_id"] + _loadable,
            )
            .set_index("donor_id")
        )

        JTBD_BASE = JTBD_BASE.join(
            _extra,
            how="left",
        )


# Same project-cost coverage guardrail used in clustering.
_project_cost_jtbd_fields = {
    "median_gift_to_project_cost_ratio_24m",
    "share_gifts_over_half_project_cost_24m",
    "share_gifts_full_project_cost_24m",
}

if "coverage_project_cost_24m" in JTBD_BASE.columns:

    _low_cov = (
        pd.to_numeric(
            JTBD_BASE["coverage_project_cost_24m"],
            errors="coerce",
        )
        .lt(0.50)
    )

    for _f in _project_cost_jtbd_fields.intersection(
        JTBD_BASE.columns
    ):
        JTBD_BASE.loc[_low_cov, _f] = np.nan


# ----------------------------------------------------------------------------
# 2. SCORE EACH JOB
#
# 1. Convert raw features to donor percentile ranks.
# 2. Orient signs consistently.
# 3. Weighted-average components within each Job.
# 4. Standardize the final Job score across donors.
# ----------------------------------------------------------------------------

JTBD_JOB_SCORES_RAW = pd.DataFrame(
    index=JTBD_BASE.index
)

_audit_rows = []


for _job, _spec in JTBD_JOB_SPECS.items():

    _available = [
        f
        for f in _spec["features"]
        if f in JTBD_BASE.columns
    ]

    _component_frame = pd.DataFrame(
        index=JTBD_BASE.index
    )

    _weights = {}


    for _f in _available:

        _x = pd.to_numeric(
            JTBD_BASE[_f],
            errors="coerce",
        )

        # Ignore unusable / constant fields.
        if (
            _x.notna().sum() < 2
            or _x.dropna().nunique() <= 1
        ):
            continue


        _signed_weight = float(
            _spec["features"][_f]
        )

        _weight = abs(
            _signed_weight
        )


        if _weight <= 0:
            continue


        _pct = _x.rank(
            method="average",
            pct=True,
        )


        if _signed_weight < 0:
            _pct = 1.0 - _pct


        _component_frame[_f] = _pct
        _weights[_f] = _weight


    _usable_components = list(
        _component_frame.columns
    )

    _min_components = int(
        _spec["min_components"]
    )


    if len(_usable_components) >= _min_components:

        _n_observed = (
            _component_frame
            .notna()
            .sum(axis=1)
        )


        _weighted_values = pd.DataFrame(
            {
                f: _component_frame[f] * _weights[f]
                for f in _usable_components
            },
            index=JTBD_BASE.index,
        )

        _observed_weights = pd.DataFrame(
            {
                f: _component_frame[f].notna().astype(float) * _weights[f]
                for f in _usable_components
            },
            index=JTBD_BASE.index,
        )


        _score = (
            _weighted_values
            .sum(
                axis=1,
                skipna=True,
            )
            /
            _observed_weights
            .sum(axis=1)
            .replace(0, np.nan)
        )


        _score = _score.where(
            _n_observed >= _min_components
        )

    else:

        _score = pd.Series(
            np.nan,
            index=JTBD_BASE.index,
        )


    JTBD_JOB_SCORES_RAW[_job] = _score


    _used_text = " | ".join(
        f"{f} ({_spec['features'][f]:+.2f})"
        for f in _usable_components
    )


    _audit_rows.append({
        "Job": _job,
        "intended status":
            "RANKABLE"
            if _spec.get("rankable", True)
            else "EVIDENCE ONLY",
        "defined features": len(
            _spec["features"]
        ),
        "usable features": len(
            _usable_components
        ),
        "features used":
            _used_text
            if _used_text
            else "—",
        "donor coverage":
            float(_score.notna().mean()),
        "definition":
            _spec["definition"],
    })


JTBD_JOB_AUDIT = pd.DataFrame(
    _audit_rows
)


# Standardize completed Job scores across donors.
JTBD_JOB_SCORES_Z = (
    JTBD_JOB_SCORES_RAW.copy()
)

for _job in JTBD_JOB_SCORES_Z.columns:

    _s = JTBD_JOB_SCORES_Z[_job]

    _sd = _s.std(
        ddof=0
    )

    if (
        pd.notna(_sd)
        and _sd > 0
    ):

        JTBD_JOB_SCORES_Z[_job] = (
            (_s - _s.mean()) / _sd
        )

    else:

        JTBD_JOB_SCORES_Z[_job] = np.nan


# ----------------------------------------------------------------------------
# 3. DETERMINE RANKABLE VS EVIDENCE-ONLY JOBS
# ----------------------------------------------------------------------------

_rankable_jobs = [

    _job

    for _job, _spec in JTBD_JOB_SPECS.items()

    if _spec.get(
        "rankable",
        True,
    )

    and (
        JTBD_JOB_SCORES_Z[_job]
        .notna()
        .mean()
        >= JTBD_MIN_JOB_COVERAGE
    )

    and (
        JTBD_JOB_SCORES_Z[_job]
        .notna()
        .sum()
        > 1
    )
]


if len(_rankable_jobs) < JTBD_TOP_N:

    raise ValueError(
        f"Only {len(_rankable_jobs)} Jobs meet the ranking rules; "
        f"cannot assign Top {JTBD_TOP_N}. "
        "Inspect JTBD_JOB_AUDIT."
    )


def _final_status(row):

    if row["intended status"] == "EVIDENCE ONLY":
        return "EVIDENCE ONLY"

    if row["Job"] in _rankable_jobs:
        return "RANKED"

    return "EVIDENCE ONLY / LOW COVERAGE"


JTBD_JOB_AUDIT[
    "Top-N status"
] = JTBD_JOB_AUDIT.apply(
    _final_status,
    axis=1,
)


_evidence_only_jobs = [

    _job

    for _job in JTBD_JOB_SCORES_Z.columns

    if _job not in _rankable_jobs
    and JTBD_JOB_SCORES_Z[_job]
        .notna()
        .sum() > 1
]


# ----------------------------------------------------------------------------
# 4. COMPLETE-CASE DONOR RANKING
#
# FIX:
# A donor enters the Top-N ranking ONLY if every rankable Job is measured.
#
# This ensures:
# - every donor is competing across the same Job set;
# - Local Stewardship missingness cannot count as "not Top-2";
# - missing Local cannot artificially make another Job easier to rank Top-2.
# ----------------------------------------------------------------------------

_measured_job_n = (

    JTBD_JOB_SCORES_Z[
        _rankable_jobs
    ]

    .notna()

    .sum(axis=1)
)


_ranking_eligible = (
    _measured_job_n
    .eq(len(_rankable_jobs))
)


JTBD_JOB_RANKS = (

    JTBD_JOB_SCORES_Z[
        _rankable_jobs
    ]

    .rank(
        axis=1,
        method="first",
        ascending=False,
    )
)


JTBD_TOPN_FLAGS = (

    JTBD_JOB_RANKS
    .le(JTBD_TOP_N)
    .where(
        _ranking_eligible,
        False,
    )
    .astype(bool)
)


JTBD_TOP1_FLAGS = (

    JTBD_JOB_RANKS
    .eq(1)
    .where(
        _ranking_eligible,
        False,
    )
    .astype(bool)
)


if _ranking_eligible.any():

    assert (

        JTBD_TOPN_FLAGS
        .loc[_ranking_eligible]
        .sum(axis=1)
        .eq(JTBD_TOP_N)
        .all()

    )


# ----------------------------------------------------------------------------
# 5. DONOR-LEVEL ASSIGNMENTS
# ----------------------------------------------------------------------------

JTBD_DONOR_ASSIGNMENTS = pd.DataFrame(
    index=JTBD_BASE.index
)

JTBD_DONOR_ASSIGNMENTS["cluster"] = (
    JTBD_BASE["cluster"]
)

JTBD_DONOR_ASSIGNMENTS[
    "jtbd_ranking_eligible"
] = _ranking_eligible

JTBD_DONOR_ASSIGNMENTS[
    "jtbd_jobs_measured"
] = _measured_job_n


for _r in range(
    1,
    JTBD_TOP_N + 1,
):

    JTBD_DONOR_ASSIGNMENTS[
        f"jtbd_top_{_r}"
    ] = JTBD_JOB_RANKS.apply(

        lambda row, r=_r:
            (
                row.index[
                    row.eq(r)
                ][0]
                if (
                    _ranking_eligible.loc[
                        row.name
                    ]
                    and row.eq(r).any()
                )
                else np.nan
            ),

        axis=1,
    )


JTBD_DONOR_ASSIGNMENTS[
    "jtbd_top_n"
] = (

    JTBD_DONOR_ASSIGNMENTS[
        [
            f"jtbd_top_{r}"
            for r in range(
                1,
                JTBD_TOP_N + 1,
            )
        ]
    ]

    .apply(
        lambda r:
            " | ".join(
                r.dropna()
                .astype(str)
            ),
        axis=1,
    )
)


# ----------------------------------------------------------------------------
# 6. CLUSTER-LEVEL TOP-N PREVALENCE
# ----------------------------------------------------------------------------

_overall_topn = (

    JTBD_TOPN_FLAGS
    .loc[_ranking_eligible]
    .mean(axis=0)
)


_overall_top1 = (

    JTBD_TOP1_FLAGS
    .loc[_ranking_eligible]
    .mean(axis=0)
)


_summary_rows = []


for _cluster in sorted(
    JTBD_BASE["cluster"]
    .dropna()
    .unique()
):

    _cluster_mask = (
        JTBD_BASE["cluster"]
        .eq(_cluster)
    )

    _eligible_mask = (
        _cluster_mask
        & _ranking_eligible
    )

    _cluster_n = int(
        _cluster_mask.sum()
    )

    _eligible_n = int(
        _eligible_mask.sum()
    )


    for _job in _rankable_jobs:

        _top_share = (

            float(
                JTBD_TOPN_FLAGS
                .loc[
                    _eligible_mask,
                    _job,
                ]
                .mean()
            )

            if _eligible_n

            else np.nan
        )


        _overall = float(
            _overall_topn[_job]
        )


        _evidence = (

            float(
                JTBD_JOB_SCORES_Z
                .loc[
                    _eligible_mask,
                    _job,
                ]
                .mean()
            )

            if _eligible_n

            else np.nan
        )


        _index = (

            _top_share / _overall

            if (
                pd.notna(_top_share)
                and _overall > 0
            )

            else np.nan
        )


        _standout = bool(

            pd.notna(_index)
            and pd.notna(_evidence)

            and _index
                >= JTBD_STANDOUT_MIN_INDEX

            and _evidence
                >= JTBD_STANDOUT_MIN_EVIDENCE_Z
        )


        _summary_rows.append({

            "cluster":
                int(_cluster),

            "cluster_n":
                _cluster_n,

            "ranking_eligible_n":
                _eligible_n,

            "ranking_eligible_share":
                (
                    _eligible_n / _cluster_n
                    if _cluster_n
                    else np.nan
                ),

            "Job":
                _job,

            f"top_{JTBD_TOP_N}_n":
                (
                    int(
                        JTBD_TOPN_FLAGS
                        .loc[
                            _eligible_mask,
                            _job,
                        ]
                        .sum()
                    )
                    if _eligible_n
                    else 0
                ),

            f"top_{JTBD_TOP_N}_share":
                _top_share,

            "overall_top_n_share":
                _overall,

            "index_vs_overall":
                _index,

            "top_1_share":
                (
                    float(
                        JTBD_TOP1_FLAGS
                        .loc[
                            _eligible_mask,
                            _job,
                        ]
                        .mean()
                    )
                    if _eligible_n
                    else np.nan
                ),

            "mean_behavioral_evidence_z":
                _evidence,

            "standout":
                _standout,

            "job_measured_share_in_cluster":
                float(
                    JTBD_JOB_SCORES_Z
                    .loc[
                        _cluster_mask,
                        _job,
                    ]
                    .notna()
                    .mean()
                ),
        })


JTBD_CLUSTER_SUMMARY = pd.DataFrame(
    _summary_rows
)


JTBD_STANDOUT_SUMMARY = (

    JTBD_CLUSTER_SUMMARY.loc[
        JTBD_CLUSTER_SUMMARY[
            "standout"
        ]
    ]

    .sort_values(
        [
            "cluster",
            "index_vs_overall",
        ],
        ascending=[
            True,
            False,
        ],
    )

    .reset_index(
        drop=True
    )
)


# ----------------------------------------------------------------------------
# 7. EVIDENCE-ONLY JOB SUMMARIES
#
# Tax Efficiency and Collective Participation live here.
# They do NOT affect donor Top-2 assignments.
# ----------------------------------------------------------------------------

_evidence_rows = []


for _cluster in sorted(
    JTBD_BASE["cluster"]
    .dropna()
    .unique()
):

    _cluster_mask = (
        JTBD_BASE["cluster"]
        .eq(_cluster)
    )


    for _job in _evidence_only_jobs:

        _x = JTBD_JOB_SCORES_Z.loc[
            _cluster_mask,
            _job,
        ]

        _evidence_rows.append({

            "cluster":
                int(_cluster),

            "Job":
                _job,

            "mean_behavioral_evidence_z":
                float(_x.mean()),

            "measured_share":
                float(_x.notna().mean()),
        })


JTBD_EVIDENCE_ONLY_SUMMARY = pd.DataFrame(
    _evidence_rows
)


# ----------------------------------------------------------------------------
# 8. BUSINESS-FRIENDLY OUTPUT
# ----------------------------------------------------------------------------

_n_eligible = int(
    _ranking_eligible.sum()
)

_n_total = len(
    _ranking_eligible
)


display(
    Markdown(
        f"""
### Jobs most reflected in observed behavior

**{len(_rankable_jobs)} Jobs currently enter the behavioral Top-{JTBD_TOP_N} ranking.**

For comparability, Top-{JTBD_TOP_N} rankings use only donors for whom **all {len(_rankable_jobs)} rankable Jobs are measurable**.

**{_n_eligible:,} of {_n_total:,} donors ({_n_eligible / _n_total:.0%})** meet that complete-case rule.

Each eligible donor contributes exactly **{JTBD_TOP_N} slots**.

A Job is labeled a **standout** for a cluster only when:

- Top-{JTBD_TOP_N} prevalence is at least **{JTBD_STANDOUT_MIN_INDEX:.2f}x overall**, and
- mean behavioral evidence is at least **+{JTBD_STANDOUT_MIN_EVIDENCE_Z:.2f} SD above average**.

These are behavioral signals consistent with Jobs — not direct evidence of donor motivation.
"""
    )
)


# -----------------------------------
# Scoring / coverage audit
# -----------------------------------

display(

    JTBD_JOB_AUDIT[
        [
            "Job",
            "Top-N status",
            "donor coverage",
            "usable features",
            "features used",
        ]
    ]

    .style

    .format({
        "donor coverage": "{:.0%}",
    })

    .hide(axis="index")

    .set_caption(
        "JTBD scoring audit — signed numbers are within-Job feature weights"
    )
)


# -----------------------------------
# Full cross-cluster Top-N matrix
# -----------------------------------

_top_col = (
    f"top_{JTBD_TOP_N}_share"
)


JTBD_TOPN_MATRIX = (

    JTBD_CLUSTER_SUMMARY

    .pivot(
        index="Job",
        columns="cluster",
        values=_top_col,
    )

    .reindex(
        _rankable_jobs
    )
)


JTBD_TOPN_MATRIX.columns = [

    f"Cluster {int(c)}"

    for c in JTBD_TOPN_MATRIX.columns
]


display(

    JTBD_TOPN_MATRIX

    .style

    .format("{:.0%}")

    .background_gradient(
        axis=None,
        cmap="Blues",
    )

    .set_caption(
        f"Share of complete-case donors in each cluster with the Job in behavioral Top {JTBD_TOP_N}"
    )
)


# -----------------------------------
# Cluster cards — standouts only
# -----------------------------------

for _cluster in sorted(
    JTBD_CLUSTER_SUMMARY[
        "cluster"
    ].unique()
):

    _all_cluster = (

        JTBD_CLUSTER_SUMMARY.loc[
            JTBD_CLUSTER_SUMMARY[
                "cluster"
            ].eq(_cluster)
        ]

        .copy()
    )


    _s = (

        _all_cluster.loc[
            _all_cluster[
                "standout"
            ]
        ]

        .sort_values(
            [
                "index_vs_overall",
                "mean_behavioral_evidence_z",
            ],
            ascending=[
                False,
                False,
            ],
        )

        .head(
            JTBD_SHOW_TOP_PER_CLUSTER
        )
    )


    _eligible_n = int(
        _all_cluster[
            "ranking_eligible_n"
        ].iloc[0]
    )

    _cluster_n = int(
        _all_cluster[
            "cluster_n"
        ].iloc[0]
    )


    if len(_s):

        _rows = []


        for _r in _s.itertuples(
            index=False
        ):

            _top_share = getattr(
                _r,
                _top_col,
            )

            _count = getattr(
                _r,
                f"top_{JTBD_TOP_N}_n",
            )


            _rows.append(

                "<tr>"

                f"<td style='padding:6px 8px;font-weight:600'>"
                f"{_r.Job}"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right'>"
                f"{_count:,} ({_top_share:.0%})"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right'>"
                f"{_r.overall_top_n_share:.0%}"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right;font-weight:600'>"
                f"{_r.index_vs_overall:.2f}x"
                f"</td>"

                f"<td style='padding:6px 8px;text-align:right;font-weight:600'>"
                f"{_r.mean_behavioral_evidence_z:+.2f} SD"
                f"</td>"

                "</tr>"
            )


        _body = f"""
        <table style="
            border-collapse:collapse;
            width:100%;
            font-size:12px;
        ">
        
            <thead>
                <tr style="background:#f8fafc">
        
                    <th style="text-align:left;padding:6px 8px">
                        Standout Job
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Cluster Top-{JTBD_TOP_N}
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Overall
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Index
                    </th>
        
                    <th style="text-align:right;padding:6px 8px">
                        Evidence vs avg
                    </th>
        
                </tr>
            </thead>
        
            <tbody>
                {''.join(_rows)}
            </tbody>
        
        </table>
        """

    else:

        _body = (
            "<div style='color:#666;font-size:13px;padding:6px 0'>"
            "No Job clears both standout thresholds. "
            "See the full Top-N matrix for weaker relative signals."
            "</div>"
        )


    display(
        HTML(
            f"""
<div style="
    border:1px solid #d1d5db;
    border-radius:8px;
    padding:12px 14px;
    margin:12px 0 18px 0;
    background:white;
">

    <div style="
        font-size:19px;
        font-weight:700;
    ">
        Cluster {int(_cluster)} — standout behavioral Job signals
    </div>

    <div style="
        color:#666;
        font-size:12px;
        margin:3px 0 9px 0;
    ">
        {_eligible_n:,} of {_cluster_n:,} donors included in
        complete-case Top-{JTBD_TOP_N} ranking
    </div>

    {_body}

</div>
"""
        )
    )


# -----------------------------------
# Evidence-only Jobs
# -----------------------------------

if len(JTBD_EVIDENCE_ONLY_SUMMARY):

    JTBD_EVIDENCE_ONLY_MATRIX = (

        JTBD_EVIDENCE_ONLY_SUMMARY

        .pivot(
            index="Job",
            columns="cluster",
            values="mean_behavioral_evidence_z",
        )
    )

    JTBD_EVIDENCE_ONLY_MATRIX.columns = [

        f"Cluster {int(c)}"

        for c in JTBD_EVIDENCE_ONLY_MATRIX.columns
    ]


    display(
        Markdown(
            """
### Evidence-only Job signals

These are useful directional signals but **cannot receive donor Top-2 slots**.

This currently includes **Tax Efficiency** and **Collective Participation**.
"""
        )
    )


    display(

        JTBD_EVIDENCE_ONLY_MATRIX

        .style

        .format("{:+.2f} SD")

        .background_gradient(
            axis=None,
            cmap="Blues",
        )

        .set_caption(
            "Evidence-only Jobs — directional behavioral signals, not Job assignments"
        )
    )


# -----------------------------------
# Jobs with no defensible score
# -----------------------------------

_skipped_text = "<br>".join(

    f"<b>{job}</b>: {reason}"

    for job, reason
    in JTBD_NOT_RANKED.items()
)


display(

    HTML(
        """
<div style="
    font-size:12px;
    color:#555;
    margin-top:12px;
">
<b>Jobs without a defensible current behavioral score:</b><br>
"""
        + _skipped_text
        +
        """
<br><br>
<i>Interpretation guardrail:</i>
these scores describe observed behaviors consistent with Jobs,
not directly observed donor motivations.
</div>
"""
    )
)


print(
    "Objects created: "
    "JTBD_DONOR_ASSIGNMENTS, "
    "JTBD_JOB_SCORES_RAW, "
    "JTBD_JOB_SCORES_Z, "
    "JTBD_JOB_RANKS, "
    "JTBD_TOPN_FLAGS, "
    "JTBD_JOB_AUDIT, "
    "JTBD_CLUSTER_SUMMARY, "
    "JTBD_STANDOUT_SUMMARY, "
    "JTBD_TOPN_MATRIX, "
    "JTBD_EVIDENCE_ONLY_SUMMARY"
)


### Jobs most reflected in observed behavior

**9 Jobs currently enter the behavioral Top-2 ranking.**

For comparability, Top-2 rankings use only donors for whom **all 9 rankable Jobs are measurable**.

**90,062 of 153,366 donors (59%)** meet that complete-case rule.

Each eligible donor contributes exactly **2 slots**.

A Job is labeled a **standout** for a cluster only when:

- Top-2 prevalence is at least **1.25x overall**, and
- mean behavioral evidence is at least **+0.20 SD above average**.

These are behavioral signals consistent with Jobs — not direct evidence of donor motivation.


Job,Top-N status,donor coverage,usable features,features used
Concrete impact,RANKED,100%,3,median_gift_to_project_cost_ratio_24m (+0.75) | share_gifts_closed_project_24m (+1.00) | share_gifts_classroom_essentials_24m (+0.50)
Directed choice,RANKED,87%,2,share_gifts_with_prior_search_24m (+1.00) | project_page_pre_gift_mean_24m (+1.00)
Giving leverage,RANKED,100%,2,share_gifts_with_match_24m (+1.00) | mean_match_excess_24m (+1.00)
Tax efficiency,EVIDENCE ONLY,100%,3,share_gifts_daf_24m (+2.00) | share_gifts_year_end_final_week_24m (+1.00) | share_gifts_q4_24m (+0.50)
Sustained giving,RANKED,100%,5,n_active_months_24m (+1.00) | entropy_gift_month_norm_24m (+1.00) | is_continuing_donor_24m (+1.00) | tenure_days_any_giving (+0.75) | monthly_longest_streak_months (+0.50)
Confidence and risk reduction,RANKED,93%,3,share_gifts_repeat_school_24m (+1.00) | share_gifts_with_prior_search_24m (+0.75) | teacher_page_pre_gift_mean_24m (+0.50)
Relational support,RANKED,100%,4,share_gifts_repeat_teacher_24m (+1.00) | top_teacher_share_count_24m (+1.00) | gifts_per_teacher_24m (+0.75) | is_teacher_referred (+0.50)
Local stewardship,RANKED,67%,2,share_gifts_within_15mi_24m (+1.00) | median_distance_mi_24m (-1.00)
Collective participation,EVIDENCE ONLY,100%,4,sharing_active_months_24m (+1.00) | share_gifts_channel_sharetray_24m (+1.00) | share_gifts_channel_facebook_24m (+0.50) | share_gifts_channel_nextdoor_24m (+0.50)
Urgent action,RANKED,100%,5,days_gift_to_expiration_median_24m (-1.00) | share_gifts_expiring_soon_24m (+1.00) | share_gifts_late_cycle_24m (+0.50) | share_gifts_same_day_first_session_24m (+0.50) | days_first_session_to_gift_median_24m (-0.50)


,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7
Job,,,,,,,
Concrete impact,8%,24%,15%,7%,14%,61%,79%
Directed choice,13%,26%,17%,14%,13%,28%,20%
Giving leverage,6%,8%,19%,58%,20%,18%,13%
Sustained giving,27%,13%,47%,14%,26%,26%,19%
Confidence and risk reduction,25%,23%,8%,17%,21%,9%,17%
Relational support,39%,19%,3%,27%,62%,14%,6%
Local stewardship,47%,56%,31%,32%,0%,0%,18%
Urgent action,14%,15%,33%,11%,15%,25%,18%
Values obligation,21%,16%,27%,21%,29%,19%,9%


Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Local stewardship,"7,724 (47%)",25%,1.89x,+0.87 SD
Relational support,"6,447 (39%)",27%,1.48x,+0.46 SD
Confidence and risk reduction,"4,054 (25%)",17%,1.44x,+0.55 SD


Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Local stewardship,"5,236 (56%)",25%,2.25x,+0.74 SD


...

Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Concrete impact,"8,831 (61%)",27%,2.23x,+0.81 SD


Standout Job,Cluster Top-2,Overall,Index,Evidence vs avg
Concrete impact,"7,058 (79%)",27%,2.92x,+1.50 SD



### Evidence-only Job signals

These are useful directional signals but **cannot receive donor Top-2 slots**.

This currently includes **Tax Efficiency** and **Collective Participation**.


,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7
Job,,,,,,,
Collective participation,+0.01 SD,-0.05 SD,+0.18 SD,+0.00 SD,+0.09 SD,-0.15 SD,-0.15 SD
Tax efficiency,-0.07 SD,-0.15 SD,+0.07 SD,-0.16 SD,-0.11 SD,+0.26 SD,+0.30 SD


Objects created: JTBD_DONOR_ASSIGNMENTS, JTBD_JOB_SCORES_RAW, JTBD_JOB_SCORES_Z, JTBD_JOB_RANKS, JTBD_TOPN_FLAGS, JTBD_JOB_AUDIT, JTBD_CLUSTER_SUMMARY, JTBD_STANDOUT_SUMMARY, JTBD_TOPN_MATRIX, JTBD_EVIDENCE_ONLY_SUMMARY
